### InMemory Vector Stores

Uses a dicctionary, and computes cosine similarity for search using numpy.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

from langchain.chat_models.base import init_chat_model
llm=init_chat_model(
    model= "gpt-4o",
    model_provider="openai"
)
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7f7237113cb0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f7236c3c830>, root_client=<openai.OpenAI object at 0x7f7237111160>, root_async_client=<openai.AsyncOpenAI object at 0x7f7236c3c590>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store

In [8]:
from langchain_core.documents import Document
sample_documents=[
    Document(
        page_content=""" The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.""",
        metadata={"source": "Wikipedia", "topic": "Great Wall of China", "page": 1}
    ),
    Document(
        page_content =""" The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower. The Eiffel Tower is one of the most recognizable structures in the world and is a global cultural icon of France.""",
        metadata={"source": "Wikipedia", "topic": "Eiffel Tower", "page": 1}
    ),
    Document(
        page_content =""" The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.""",
        metadata={"source": "Wikipedia", "topic": "Pyramids of Giza", "page": 1}
    ),
    Document(
        page_content =""" The Taj Mahal is an ivory-white marble mausoleum located in Agra, India. It was commissioned by the Mughal emperor Shah Jahan in memory of his favorite wife, Mumtaz Mahal. The Taj Mahal is widely recognized as a symbol of love and is one of the most famous buildings in the world.""",
        metadata={"source": "Wikipedia", "topic": "Taj Mahal", "page": 1}   
    ),
    Document(
        page_content =""" The Statue of Liberty is a colossal neoclassical sculpture on Liberty Island in New York Harbor, USA. It was a gift from the people of France to the United States and was designed by French sculptor Frédéric Auguste Bartholdi. The statue is a symbol of freedom and democracy and is one of the most iconic landmarks in the world.""",
        metadata={"source": "Wikipedia", "topic": "Statue of Liberty", "page": 1}
    )
]

In [9]:
vector_store.add_documents(sample_documents)

['d0187867-7ea3-4614-97ac-95af15c9b0b4',
 'dc3738c1-3fab-45d0-950e-e4e359e9f331',
 '28b3f03d-45bc-42c4-b8be-5a2433368eaa',
 'a463f0de-450d-4c28-9183-6e6eae3f889f',
 '00d5aa6a-b8d5-4378-9a3e-44f2db2c8a1e']

In [11]:
vector_store.similarity_search("Where is the Great Wall located?", k=2)

[Document(id='d0187867-7ea3-4614-97ac-95af15c9b0b4', metadata={'source': 'Wikipedia', 'topic': 'Great Wall of China', 'page': 1}, page_content=' The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.'),
 Document(id='28b3f03d-45bc-42c4-b8be-5a2433368eaa', metadata={'source': 'Wikipedia', 'topic': 'Pyramids of Giza', 'page': 1}, page_content=' The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.')]

In [14]:
### vector store to retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x7f723e0bce10>, search_kwargs={'k': 2})

In [15]:
##Invoke
retriever.invoke("Where is the Great Wall located?")

[Document(id='d0187867-7ea3-4614-97ac-95af15c9b0b4', metadata={'source': 'Wikipedia', 'topic': 'Great Wall of China', 'page': 1}, page_content=' The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.'),
 Document(id='28b3f03d-45bc-42c4-b8be-5a2433368eaa', metadata={'source': 'Wikipedia', 'topic': 'Pyramids of Giza', 'page': 1}, page_content=' The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.')]